#### Start Spark Session

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/26 17:45:20 WARN Utils: Your hostname, codespaces-91f0b4, resolves to a loopback address: 127.0.0.1; using 10.0.3.125 instead (on interface eth0)
26/02/26 17:45:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 17:45:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Read Green Taxi Parquet Files

In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

#### Rename Green Taxi Date Columns

In [3]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

#### Read Yellow Taxi Parquet Files

In [4]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

#### Rename Yellow Taxi Date Columns

In [5]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

#### Find Common Columns Between Green and Yellow

common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

#### Select Common Columns and Add Service Type

In [7]:
from pyspark.sql import functions as F

df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))

#### Combine Green and Yellow into One DataFrame

In [8]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

#### Count Records by Service Type

In [11]:
df_trips_data.groupBy('service_type').count().show()

[Stage 3:===============================================>           (4 + 1) / 5]

+------------+-------+
|service_type|  count|
+------------+-------+
|       green|  76518|
|      yellow|7774773|
+------------+-------+



#### Check Combined Columns

In [12]:
df_trips_data.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'service_type']

#### Register DataFrame as SQL Table

In [13]:
df_trips_data.registerTempTable('trips_data')

/workspaces/data-engineering-zoomcamp/06-batch/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


#### Run a Simple SQL Query to Verify

In [14]:
spark.sql("""
SELECT
    service_type,
    count(1)
FROM
    trips_data
GROUP BY 
    service_type
""").show()

[Stage 6:===============================================>           (4 + 1) / 5]

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green|   76518|
|      yellow| 7774773|
+------------+--------+



#### Run the Full Revenue SQL Query

In [15]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [18]:
df_result.show()

[Stage 12:==================================>                       (3 + 2) / 5]

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|         254

#### Save Results to Parquet

In [16]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')

#### Verify Output Files

In [17]:
!ls -lh data/report/revenue/

total 56K
-rw-r--r-- 1 codespace codespace   0 Feb 26 17:51 _SUCCESS
-rw-r--r-- 1 codespace codespace 56K Feb 26 17:51 part-00000-f1b59ee4-632c-47ee-818c-75397e4f4d73-c000.snappy.parquet
